# SARIMA — Baseline Clássico de Série Temporal

Modelo estatístico de referência para previsão de séries temporais (SARIMA), aplicado à série de casos do DF. Serve como *baseline* clássico consagrado, ausente até aqui no conjunto de modelos. Ajusta-se à série de treino de cada cenário e projeta o período de teste, com intervalos de predição a partir da distribuição preditiva do modelo. Kernel: `qml_dengue` (requer statsmodels).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
sys.path.insert(0, os.path.abspath(".."))
from feature_engineering import construir_features, splits_validacao
from utils_qml import metricas, salvar_padrao, plot_pred, validar_json_saida

CACHE = os.path.join(REPO_ROOT, "data", "dados_dengue_df_real.json")
with open(CACHE, encoding="utf-8") as f:
    dados_brutos = json.load(f)
dataset = construir_features(dados_brutos, n_lags=4)
splits  = splits_validacao(dataset)

NOMES = {0: "C1", 1: "C2", 2: "C3"}
CENARIOS = {}
for idx, split in enumerate(splits[:3]):
    tr, te = split["treino"], split["teste"]
    CENARIOS[NOMES[idx]] = {"y_train": np.array(tr["y"]), "y_test": np.array(te["y"])}
print("Cenários:", {c: (len(d['y_train']), len(d['y_test'])) for c, d in CENARIOS.items()})

In [ ]:
# ── Ajuste SARIMA por cenário e projeção do teste ──
np.random.seed(42)
B = 30            # amostras da distribuição preditiva -> intervalos p/ WIS
RESULTADOS = {}

for cen, d in CENARIOS.items():
    t0 = time.time()
    ytr, yte = d["y_train"], d["y_test"]
    ylog = np.log1p(ytr.astype(float))
    n = len(ylog)
    seasonal = (1, 0, 0, 52) if n >= 60 else (0, 0, 0, 0)   # sazonal anual só com histórico suficiente
    try:
        res = SARIMAX(ylog, order=(2, 1, 2), seasonal_order=seasonal,
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=300)
    except Exception:
        res = SARIMAX(ylog, order=(1, 1, 1)).fit(disp=False, maxiter=300)

    fc = res.get_forecast(steps=len(yte))
    mean_log = np.asarray(fc.predicted_mean)
    se_log   = np.asarray(fc.se_mean)
    se_log   = np.minimum(se_log, 1.0)   # limita incerteza log (evita intervalos explosivos em horizonte longo)
    rng = np.random.RandomState(42)
    amostras_log = rng.normal(mean_log[None, :], se_log[None, :], size=(B, len(yte)))
    amostras_log = np.clip(amostras_log, 0.0, np.log1p(100000))   # teto epidemiológico
    preds = np.maximum(np.expm1(amostras_log), 0.0)
    med = np.maximum(np.expm1(mean_log), 0.0)

    m = metricas(yte, med, preds, nome=f"SARIMA_{cen}")
    RESULTADOS[cen] = {**m, "preds_matrix": preds, "mediana": med, "y_test": yte, "tempo_s": time.time() - t0}
    print(f"{cen}: R2={m['R2']:.4f} | RMSE={m['RMSE']:.1f} | WIS={m['WIS']:.2f} | sazonal={seasonal} | {RESULTADOS[cen]['tempo_s']:.0f}s")

In [ ]:
# ── Tabela + figura ──
print(f"\n{'='*60}")
print(f"{'SARIMA — Baseline Clássico de Série Temporal':^60}")
print(f"{'='*60}")
print(f"{'Cenário':<10}{'R²':>10}{'RMSE':>12}{'WIS':>12}")
print("-"*60)
for cen, r in RESULTADOS.items():
    print(f"{cen:<10}{r['R2']:>10.4f}{r['RMSE']:>12.1f}{r['WIS']:>12.2f}")
print("="*60)

plot_pred(RESULTADOS,
          "SARIMA: Predição vs. Observado (dados reais DF 2022-2025)",
          "sarima_pred_vs_obs.png")

In [ ]:
SCHEMA_INFO = {
    "algoritmo": "SARIMA",
    "fase": 41,
    "tipo": "classico_serie_temporal",
    "n_parametros_quanticos": None,
    "config": {"order": "(2,1,2)", "seasonal": "(1,0,0,52) quando n>=60", "alvo": "log1p", "n_amostras_intervalo": B},
}
doc = salvar_padrao(RESULTADOS, SCHEMA_INFO)
validar_json_saida(doc, contexto="SARIMA")